# LeetCode 3258 — Count Substrings That Satisfy K-Constraint I bằng Python

Link đề: https://leetcode.com/problems/count-substrings-that-satisfy-k-constraint-i/ · Độ khó: **Easy** · Chủ đề: String, Sliding Window (cửa sổ trượt)

Skeleton LeetCode:

```python
class Solution:
    def countKConstraintSubstrings(self, s: str, k: int) -> int:
        ...
```

Ghi chú: ba ví dụ chính lấy từ đề; những ví dụ tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc. Mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả đối chiếu hàng nghìn chuỗi ngẫu nhiên với bản vét cạn).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho chuỗi **nhị phân** `s` (chỉ `'0'` và `'1'`) và số nguyên `k`; một chuỗi thỏa **k-constraint** nếu **số `0` ≤ k** **hoặc** **số `1` ≤ k**; đếm số **chuỗi con** (substring, liên tiếp, không rỗng) của `s` thỏa điều kiện đó.
2. Nắm các công cụ Python: **cắt chuỗi** `s[i:j]`, `str.count`, `enumerate`, **mảng đếm hai ô** `cnt[0]`, `cnt[1]` truy cập bằng `int(ch)`, **tổng tiền tố** (`itertools.accumulate`), `bisect` và vòng `while` co cửa sổ.
3. Đi từ **vét cạn mọi chuỗi con và đếm bằng `count`** (`O(n³)`) → **mở rộng bên phải và cập nhật đếm dần** (`O(n²)`) → **tổng tiền tố + tìm kiếm nhị phân** (`O(n log n)`) → **cửa sổ trượt hai con trỏ** (`O(n)`) → **với mỗi đầu trái, con trỏ phải chỉ đi tới**.
4. Hiểu mấu chốt: điều kiện k-constraint có tính **"cha hợp lệ thì con hợp lệ"** (bỏ bớt ký tự chỉ làm số `0`, số `1` giảm) nên với mỗi đầu phải `r`, các đầu trái hợp lệ tạo thành một **đoạn liên tục kết thúc ở `r`**, và đầu trái nhỏ nhất `l` **không bao giờ lùi** khi `r` tăng — đó là cơ sở của **cửa sổ trượt**.
5. Tránh các lỗi kinh điển: nhầm **`and` / `or`** (hợp lệ là `cnt0 <= k` **hoặc** `cnt1 <= k`, nên *vi phạm* là cả hai `> k`), đếm `r - l` thay vì `r - l + 1`, co cửa sổ bằng `if` thay vì `while`, và quên trừ đúng ký tự bên trái khi `l` tiến.

## 2. Đề bài & ví dụ

Cho **chuỗi nhị phân** `s` và một số nguyên `k`.

Một chuỗi nhị phân thỏa **k-constraint** nếu **một trong hai** điều kiện sau đúng:

- Số ký tự `'0'` trong chuỗi **không vượt quá** `k`.
- Số ký tự `'1'` trong chuỗi **không vượt quá** `k`.

Hãy trả về **số chuỗi con** (substring) của `s` thỏa **k-constraint**.

**Ví dụ 1**

- Input: `s = "10101"`, `k = 1`
- Output: `12`
- Giải thích: mọi chuỗi con của `s` đều thỏa, **trừ** `"1010"`, `"10101"` và `"0101"`.

**Ví dụ 2**

- Input: `s = "1010101"`, `k = 2`
- Output: `25`
- Giải thích: mọi chuỗi con thỏa, **trừ** những chuỗi con có độ dài lớn hơn `5`.

**Ví dụ 3**

- Input: `s = "11111"`, `k = 1`
- Output: `15`
- Giải thích: mọi chuỗi con đều thỏa (không có `'0'` nào, số `0` bằng `0 <= 1`).

**Ràng buộc:** `1 <= s.length <= 50`; `1 <= k <= s.length`; `s[i]` là `'0'` hoặc `'1'`.

In [1]:
from typing import List
from collections import Counter
from itertools import accumulate
from bisect import bisect_right
import random

examples = [
    ("10101", 1, 12),
    ("1010101", 2, 25),
    ("11111", 1, 15),
]
for s, k, expected in examples:
    print(f"s={s!r}, k={k} -> kỳ vọng {expected}")

s='10101', k=1 -> kỳ vọng 12
s='1010101', k=2 -> kỳ vọng 25
s='11111', k=1 -> kỳ vọng 15


## 3. Kiến thức nền

### 3.1. Chuỗi con (substring) và cắt chuỗi `s[i:j]`

Chuỗi con là một đoạn **liên tiếp** của `s`. `s[i:j]` lấy các ký tự có chỉ số `i, i+1, ..., j-1` (**bao gồm `i`, không bao gồm `j`**). Chuỗi dài `n` có `n * (n + 1) / 2` chuỗi con không rỗng (mỗi cặp đầu–cuối `l <= r` cho một chuỗi con).

In [2]:
s = "10101"
print(s[0:3], s[1:2], s[2:5])               # 101 0 101
print("đếm chuỗi con:", len(s) * (len(s) + 1) // 2)      # 15
subs = [s[l:r + 1] for l in range(len(s)) for r in range(l, len(s))]
print(len(subs), subs)

101 0 101
đếm chuỗi con: 15
15 ['1', '10', '101', '1010', '10101', '0', '01', '010', '0101', '1', '10', '101', '0', '01', '1']


### 3.2. `str.count(ch)` — đếm số lần xuất hiện

`t.count("0")` đếm số ký tự `"0"` trong chuỗi `t` (quét hết chuỗi, `O(độ dài)`). Với chuỗi nhị phân, số `1` bằng `len(t) - t.count("0")`.

In [3]:
t = "1010"
print(t.count("0"), t.count("1"), len(t) - t.count("0"))    # 2 2 2

2 2 2


### 3.3. Đếm bằng mảng hai ô `cnt[int(ch)]`

Với ký tự `'0'`/`'1'`, `int(ch)` cho `0`/`1`, nên có thể dùng danh sách `cnt = [0, 0]`: `cnt[0]` là số `'0'`, `cnt[1]` là số `'1'`. Thêm ký tự `ch`: `cnt[int(ch)] += 1`; bớt đi: `cnt[int(ch)] -= 1`. Nhờ đó **cập nhật đếm trong `O(1)`** mà không cần quét lại.

In [4]:
cnt = [0, 0]
for ch in "10110":
    cnt[int(ch)] += 1
print(cnt)                 # [2, 3]: hai số 0, ba số 1
cnt[int("1")] -= 1         # bớt một ký tự '1'
print(cnt)                 # [2, 2]

[2, 3]
[2, 2]


### 3.4. Hợp lệ khi `min(zeros, ones) <= k`

Điều kiện là `zeros <= k` **hoặc** `ones <= k`. Hai vế "hoặc" tương đương với `min(zeros, ones) <= k` (nếu số nhỏ hơn đã `<= k` thì ít nhất một điều kiện đúng; và ngược lại). Phủ định: **vi phạm** khi `zeros > k` **và** `ones > k` (cả hai cùng lớn hơn `k`). Quy tắc De Morgan: `not (A or B) == (not A) and (not B)`.

In [5]:
k = 1
for zeros, ones in [(0, 5), (2, 1), (1, 1), (2, 2), (3, 7)]:
    ok_or = zeros <= k or ones <= k
    ok_min = min(zeros, ones) <= k
    bad = zeros > k and ones > k
    print(f"zeros={zeros}, ones={ones}: hợp lệ={ok_or}, min<=k={ok_min}, vi phạm={bad}")

zeros=0, ones=5: hợp lệ=True, min<=k=True, vi phạm=False
zeros=2, ones=1: hợp lệ=True, min<=k=True, vi phạm=False
zeros=1, ones=1: hợp lệ=True, min<=k=True, vi phạm=False
zeros=2, ones=2: hợp lệ=False, min<=k=False, vi phạm=True
zeros=3, ones=7: hợp lệ=False, min<=k=False, vi phạm=True


### 3.5. Tổng tiền tố (prefix sum) với `itertools.accumulate`

Nếu `pre[i]` = số `'0'` trong `s[0:i]` thì số `'0'` trong `s[l:r+1]` là `pre[r+1] - pre[l]`. `accumulate` tạo mảng tiền tố; ta thêm `0` ở đầu bằng `initial=0`. Mỗi lần hỏi đếm trong đoạn tốn `O(1)`.

In [6]:
s = "10101"
zeros = list(accumulate((1 if ch == "0" else 0 for ch in s), initial=0))
print(zeros)                          # [0, 0, 1, 1, 2, 2]
l, r = 1, 3                           # s[1:4] = "010"
print(s[l:r + 1], "có", zeros[r + 1] - zeros[l], "số 0")      # 2

[0, 0, 1, 1, 2, 2]
010 có 2 số 0


### 3.6. `bisect_right` — tìm vị trí chèn trên dãy không giảm

`bisect_right(a, x)` trên danh sách **đã sắp tăng (không giảm)** `a` trả về số phần tử `<= x` (chỉ số chèn `x` sang bên phải). Tốn `O(log n)`. Ta sẽ dùng nó để tìm **đầu phải xa nhất** thỏa điều kiện khi đã biết dãy đếm là không giảm.

In [7]:
a = [0, 0, 1, 1, 2, 2]
print(bisect_right(a, 1))      # 4  (có bốn phần tử <= 1)
print(bisect_right(a, 5))      # 6
print(bisect_right(a, -1))     # 0

4
6
0


### 3.7. Cửa sổ trượt `[l, r]` và vòng `while`

**Cửa sổ trượt** (sliding window): giữ hai chỉ số `l <= r` biểu diễn đoạn `s[l..r]`. Mỗi bước **mở rộng `r` thêm một**, rồi trong khi cửa sổ **chưa hợp lệ** thì **co `l` lên** (`l += 1`). Mỗi chỉ số chỉ được thêm một lần và bớt một lần ⇒ tổng `O(n)`. Dùng `while` (lặp tới khi hết vi phạm) chứ không phải `if`.

In [8]:
# Mô hình hóa đơn giản: giữ cửa sổ có tổng <= 5 trên dãy dương
nums = [2, 3, 1, 4, 1]
l = 0
total = 0
for r, x in enumerate(nums):
    total += x
    while total > 5:
        total -= nums[l]
        l += 1
    print(f"r={r}: cửa sổ nums[{l}..{r}] = {nums[l:r + 1]}, tổng = {total}")

r=0: cửa sổ nums[0..0] = [2], tổng = 2
r=1: cửa sổ nums[0..1] = [2, 3], tổng = 5
r=2: cửa sổ nums[1..2] = [3, 1], tổng = 4
r=3: cửa sổ nums[2..3] = [1, 4], tổng = 5
r=4: cửa sổ nums[3..4] = [4, 1], tổng = 5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Làm Ví dụ 1 bằng tay: `s = "10101"`, `k = 1`. Chuỗi con và số `0`/`1` của chúng (liệt kê theo đầu trái `l`):

| `l` | chuỗi con | (số `0`, số `1`) | hợp lệ? |
|---|---|---|---|
| 0 | `1` | (0,1) | có |
| 0 | `10` | (1,1) | có |
| 0 | `101` | (1,2) | có (`0` chỉ `1 <= 1`) |
| 0 | `1010` | (2,2) | **không** (cả hai `> 1`) |
| 0 | `10101` | (2,3) | **không** |
| 1 | `0`, `01`, `010` | (1,0), (1,1), (2,1)... | có cả ba (`010`: một `1`) |
| 1 | `0101` | (2,2) | **không** |
| 2 | `1`, `10`, `101` | | có cả ba |
| 3 | `0`, `01` | | có cả hai |
| 4 | `1` | | có |

Tổng `15` chuỗi con, `3` cái không hợp lệ ⇒ `12`.

**Hai quan sát quan trọng:**

1. Với mỗi đầu trái `l`, các chuỗi con hợp lệ là `s[l..l], s[l..l+1], ...` **dừng lại** ở đâu đó rồi **không bao giờ hợp lệ trở lại** (đã vi phạm thì thêm ký tự chỉ càng vi phạm, vì số `0` và `1` không giảm).
2. Đầu trái `l = 0` dừng ở độ dài `3` (`101`), `l = 1` dừng ở `010` (độ dài `3`) — tức đầu phải xa nhất tăng dần `2, 3, 4`.

## 5. Cách 1 — Brute Force / baseline: liệt kê mọi chuỗi con và `count`

**Ý tưởng.** Hai vòng lặp chọn `l`, `r`; cắt `t = s[l:r+1]`, đếm `t.count("0")` và `t.count("1")`, kiểm tra `<= k` rồi cộng một nếu hợp lệ. Đúng từng chữ của đề nên rất dễ tin cậy — dùng làm **bản chuẩn đối chiếu** cho các cách nhanh hơn.

In [9]:
def count_brute(s: str, k: int) -> int:
    n = len(s)
    ans = 0
    for l in range(n):
        for r in range(l, n):
            t = s[l:r + 1]
            if t.count("0") <= k or t.count("1") <= k:
                ans += 1
    return ans


for s, k, expected in examples:
    print(s, k, "->", count_brute(s, k), "(kỳ vọng", expected, ")")

10101 1 -> 12 (kỳ vọng 12 )
1010101 2 -> 25 (kỳ vọng 25 )
11111 1 -> 15 (kỳ vọng 15 )


**Độ phức tạp.** `O(n²)` cặp `(l, r)`, mỗi cặp cắt và đếm `O(n)` ⇒ `O(n³)` time, `O(n)` space (chuỗi con tạm). Với `n <= 50` là cỡ `~20 000` thao tác — **đủ nhanh**, nhưng ta học cách tốt hơn vì ý tưởng sẽ dùng lại cho các bài có `n` lớn (bản II của bài này có `n` tới `10^5`).

**Nút thắt.** Ta đếm lại từ đầu cho mỗi cặp `(l, r)`, dù `s[l..r+1]` chỉ hơn `s[l..r]` đúng **một** ký tự. Thay vì đếm lại, hãy **cập nhật dần**.

## 6. Ý tưởng thuật toán / bước đột phá

**Tính đơn điệu.** Nếu chuỗi `t` thỏa k-constraint thì **mọi chuỗi con của `t`** cũng thỏa (bỏ ký tự chỉ làm số `0` và số `1` giảm hoặc giữ nguyên, nên điều kiện `<= k` vẫn đúng). Hệ quả:

- Cố định đầu phải `r`. Nếu `s[l..r]` hợp lệ thì **mọi** `s[l'..r]` với `l <= l' <= r` cũng hợp lệ. Vậy các `l` hợp lệ là một đoạn `[L(r), r]`, và số chuỗi con hợp lệ kết thúc tại `r` là **`r - L(r) + 1`**.
- `L(r)` **không giảm** khi `r` tăng: nếu `s[l..r+1]` hợp lệ thì `s[l..r]` (con của nó) cũng hợp lệ, nên `L(r) <= L(r+1)`.

Vì `L` chỉ tiến lên, ta dùng **hai con trỏ**: tăng `r`, và đẩy `l` lên *chỉ khi cần*. Đáp án là tổng `r - l + 1` qua mọi `r`.

Công thức nhanh hơn một bậc từ `O(n³)` xuống `O(n)` nhờ hai ý: **cập nhật đếm theo `O(1)`** và **không bao giờ lùi `l`**.

In [10]:
# Kiểm chứng tính đơn điệu bằng thực nghiệm: nếu t hợp lệ thì mọi chuỗi con của t hợp lệ
def valid(t: str, k: int) -> bool:
    return t.count("0") <= k or t.count("1") <= k


rng = random.Random(1)
for _ in range(300):
    n = rng.randint(1, 10)
    s = "".join(rng.choice("01") for _ in range(n))
    k = rng.randint(1, n)
    for l in range(n):
        for r in range(l, n):
            if valid(s[l:r + 1], k):
                # mọi chuỗi con bên trong cũng hợp lệ
                for a in range(l, r + 1):
                    for b in range(a, r + 1):
                        assert valid(s[a:b + 1], k)
print("tính đơn điệu: PASS")

tính đơn điệu: PASS


## 7. Các cách tiếp theo

Phần này có **4 cách** nữa (cùng Cách 1 là 5 cách). Mỗi cách: giải thích → code demo → code nộp LeetCode → độ phức tạp.

### Cách 2 — Mở rộng bên phải, cập nhật đếm dần, có `break`

Giải thích từng đoạn:

- Với mỗi đầu trái `l`, đặt `cnt = [0, 0]`.
- Với `r` từ `l` trở đi, thêm `s[r]` vào đếm (`O(1)`).
- Nếu **vi phạm** (`cnt[0] > k` **và** `cnt[1] > k`) thì `break`: các `r` lớn hơn chắc chắn vi phạm (tính đơn điệu). Ngược lại `ans += 1`.

In [11]:
def count_extend(s: str, k: int) -> int:
    n = len(s)
    ans = 0
    for l in range(n):
        cnt = [0, 0]
        for r in range(l, n):
            cnt[int(s[r])] += 1
            if cnt[0] > k and cnt[1] > k:      # vi phạm -> các r sau cũng vi phạm
                break
            ans += 1
    return ans


for s, k, expected in examples:
    assert count_extend(s, k) == expected
print("Cách 2 chạy đúng trên các ví dụ")

Cách 2 chạy đúng trên các ví dụ


In [12]:
# Code nộp LeetCode — Cách 2
class Solution:
    def countKConstraintSubstrings(self, s: str, k: int) -> int:
        n = len(s)
        ans = 0
        for l in range(n):
            cnt = [0, 0]
            for r in range(l, n):
                cnt[int(s[r])] += 1
                if cnt[0] > k and cnt[1] > k:
                    break
                ans += 1
        return ans


for s, k, expected in examples:
    assert Solution().countKConstraintSubstrings(s, k) == expected
print("OK")

OK


**Độ phức tạp Cách 2:** Time `O(n²)` (hai vòng lặp, thêm `break`), Space `O(1)`.

### Cách 3 — Tổng tiền tố + tìm kiếm nhị phân đầu phải xa nhất

Giải thích từng đoạn:

- Dựng `z[i]` = số `0` trong `s[0:i]` và `o[i]` = số `1` trong `s[0:i]`.
- Với đầu trái `l`, số `0` trong `s[l..r]` là `z[r+1] - z[l]`: dãy này **không giảm** theo `r`; số `1` cũng vậy.
- `s[l..r]` hợp lệ khi `z[r+1] - z[l] <= k` **hoặc** `o[r+1] - o[l] <= k`. Mỗi vế cho một **giới hạn trên của `r`** (nhờ dãy không giảm): `r1` lớn nhất thỏa vế đầu, `r2` lớn nhất thỏa vế sau. Hợp lệ khi `r <= max(r1, r2)`.
- Đặt `j = r + 1` (chỉ số tiền tố). `bisect_right(z, z[l] + k)` cho số phần tử của `z` mà `<= z[l] + k`; trừ `1` được **chỉ số `j` lớn nhất** thỏa `z[j] - z[l] <= k` (gọi là `jz`). Tương tự `jo` cho số `1`.
- Các `j` hợp lệ là `l + 1, ..., max(jz, jo)` (cắt tối đa ở `n`), nên có `max(jz, jo) - l` chuỗi con hợp lệ bắt đầu ở `l`.

In [13]:
def count_prefix_bisect(s: str, k: int) -> int:
    n = len(s)
    z = list(accumulate((1 if ch == "0" else 0 for ch in s), initial=0))   # z[i]: số 0 trong s[:i]
    o = list(accumulate((1 if ch == "1" else 0 for ch in s), initial=0))   # o[i]: số 1 trong s[:i]
    ans = 0
    for l in range(n):
        # j = r + 1 lớn nhất sao cho z[j] - z[l] <= k (z không giảm nên bisect hợp lệ)
        jz = bisect_right(z, z[l] + k) - 1
        jo = bisect_right(o, o[l] + k) - 1
        j = min(n, max(jz, jo))
        ans += j - l                        # các j = l+1, ..., j  <->  r = l, ..., j-1
    return ans


for s, k, expected in examples:
    print(s, k, "->", count_prefix_bisect(s, k), "(kỳ vọng", expected, ")")
    assert count_prefix_bisect(s, k) == expected

10101 1 -> 12 (kỳ vọng 12 )
1010101 2 -> 25 (kỳ vọng 25 )
11111 1 -> 15 (kỳ vọng 15 )


In [14]:
# Code nộp LeetCode — Cách 3
class Solution:
    def countKConstraintSubstrings(self, s: str, k: int) -> int:
        n = len(s)
        z = list(accumulate((1 if ch == "0" else 0 for ch in s), initial=0))
        o = list(accumulate((1 if ch == "1" else 0 for ch in s), initial=0))
        ans = 0
        for l in range(n):
            jz = bisect_right(z, z[l] + k) - 1
            jo = bisect_right(o, o[l] + k) - 1
            ans += min(n, max(jz, jo)) - l
        return ans


for s, k, expected in examples:
    assert Solution().countKConstraintSubstrings(s, k) == expected
print("OK")

OK


**Độ phức tạp Cách 3:** Time `O(n log n)` (dựng tiền tố `O(n)`, mỗi `l` hai lần `bisect`), Space `O(n)`. Chậm hơn cửa sổ trượt nhưng minh họa rõ tính đơn điệu của đếm.

### Cách 4 — Cửa sổ trượt hai con trỏ (cách chuẩn)

Giải thích từng đoạn:

- `cnt = [0, 0]` đếm số `0`, số `1` trong cửa sổ `s[l..r]`; `l = 0`, `ans = 0`.
- Với mỗi `r`: thêm `s[r]` vào `cnt`.
- `while cnt[0] > k and cnt[1] > k`: cửa sổ vi phạm; bớt `s[l]` khỏi `cnt` rồi `l += 1`. Dừng khi hợp lệ trở lại.
- Mọi `l'` trong `[l, r]` cho chuỗi con hợp lệ kết thúc ở `r`, nên `ans += r - l + 1`.

In [15]:
def count_window(s: str, k: int) -> int:
    cnt = [0, 0]
    ans = l = 0
    for r, ch in enumerate(s):
        cnt[int(ch)] += 1
        while cnt[0] > k and cnt[1] > k:       # cửa sổ vi phạm: co từ bên trái
            cnt[int(s[l])] -= 1
            l += 1
        ans += r - l + 1                       # số chuỗi con hợp lệ kết thúc ở r
    return ans


for s, k, expected in examples:
    assert count_window(s, k) == expected
print("Cách 4 chạy đúng trên các ví dụ")

Cách 4 chạy đúng trên các ví dụ


In [16]:
# Code nộp LeetCode — Cách 4
class Solution:
    def countKConstraintSubstrings(self, s: str, k: int) -> int:
        cnt = [0, 0]
        ans = l = 0
        for r, ch in enumerate(s):
            cnt[int(ch)] += 1
            while cnt[0] > k and cnt[1] > k:
                cnt[int(s[l])] -= 1
                l += 1
            ans += r - l + 1
        return ans


for s, k, expected in examples:
    assert Solution().countKConstraintSubstrings(s, k) == expected
print("OK")

OK


**Độ phức tạp Cách 4:** Time `O(n)` (mỗi chỉ số vào cửa sổ một lần, ra một lần), Space `O(1)`.

### Cách 5 — Với mỗi đầu trái `l`, con trỏ phải chỉ đi tới

Giải thích từng đoạn: đảo vai trò hai con trỏ so với Cách 4. Ta duyệt `l` từ `0`; con trỏ `r` là **vị trí kế tiếp chưa thêm** vào cửa sổ, **không bao giờ lùi**. Trước khi xét `l`, kéo `r` tới khi thêm `s[r]` sẽ gây vi phạm; khi đó cửa sổ `s[l..r-1]` là đoạn hợp lệ dài nhất bắt đầu từ `l`, đóng góp `r - l` chuỗi con. Sau đó bớt `s[l]` khỏi đếm để chuyển sang `l + 1`.

In [17]:
def count_left_driven(s: str, k: int) -> int:
    n = len(s)
    cnt = [0, 0]
    r = 0
    ans = 0
    for l in range(n):
        # mở rộng r chừng nào thêm s[r] vẫn hợp lệ
        while r < n:
            cnt[int(s[r])] += 1
            if cnt[0] > k and cnt[1] > k:
                cnt[int(s[r])] -= 1            # thêm vào gây vi phạm -> hoàn lại và dừng
                break
            r += 1
        ans += r - l                           # s[l..r-1] là đoạn hợp lệ dài nhất từ l
        cnt[int(s[l])] -= 1                    # chuyển sang l + 1
    return ans


for s, k, expected in examples:
    assert count_left_driven(s, k) == expected
print("Cách 5 chạy đúng trên các ví dụ")

Cách 5 chạy đúng trên các ví dụ


In [18]:
# Code nộp LeetCode — Cách 5
class Solution:
    def countKConstraintSubstrings(self, s: str, k: int) -> int:
        n = len(s)
        cnt = [0, 0]
        r = 0
        ans = 0
        for l in range(n):
            while r < n:
                cnt[int(s[r])] += 1
                if cnt[0] > k and cnt[1] > k:
                    cnt[int(s[r])] -= 1
                    break
                r += 1
            ans += r - l
            cnt[int(s[l])] -= 1
        return ans


for s, k, expected in examples:
    assert Solution().countKConstraintSubstrings(s, k) == expected
print("OK")

OK


**Độ phức tạp Cách 5:** Time `O(n)` (mỗi chỉ số của `r` được thêm một lần; mỗi lần dừng chỉ hoàn lại một thao tác), Space `O(1)`. Lưu ý một chi tiết: khi `r == l` (cửa sổ rỗng) việc thêm `s[l]` không bao giờ vi phạm vì `k >= 1`, nên `r > l` luôn đúng sau khi mở rộng.

## 8. TRACE từng bước

Ta trace Cách 4 (cửa sổ trượt) trên Ví dụ 1 và trên một ví dụ tự dựng khó hơn `s = "0011100"`, `k = 1` (tự dựng; đáp án kiểm bằng vét cạn ngay trong cell). Cách 5 được trace trên Ví dụ 1 để thấy con trỏ phải chỉ đi tới.

In [19]:
def trace_window(s: str, k: int) -> int:
    print(f"s = {s!r}, k = {k}")
    cnt = [0, 0]
    ans = l = 0
    for r, ch in enumerate(s):
        cnt[int(ch)] += 1
        print(f"  r={r} thêm {ch!r}: cnt(0,1) = {tuple(cnt)}", end="")
        while cnt[0] > k and cnt[1] > k:
            cnt[int(s[l])] -= 1
            print(f" | vi phạm -> bỏ s[{l}]={s[l]!r}, cnt = {tuple(cnt)}", end="")
            l += 1
        ans += r - l + 1
        print(f" | l={l}, cửa sổ {s[l:r + 1]!r}, +{r - l + 1} -> ans = {ans}")
    return ans


assert trace_window("10101", 1) == 12
assert trace_window("0011100", 1) == count_brute("0011100", 1)

s = '10101', k = 1
  r=0 thêm '1': cnt(0,1) = (0, 1) | l=0, cửa sổ '1', +1 -> ans = 1
  r=1 thêm '0': cnt(0,1) = (1, 1) | l=0, cửa sổ '10', +2 -> ans = 3
  r=2 thêm '1': cnt(0,1) = (1, 2) | l=0, cửa sổ '101', +3 -> ans = 6
  r=3 thêm '0': cnt(0,1) = (2, 2) | vi phạm -> bỏ s[0]='1', cnt = (2, 1) | l=1, cửa sổ '010', +3 -> ans = 9
  r=4 thêm '1': cnt(0,1) = (2, 2) | vi phạm -> bỏ s[1]='0', cnt = (1, 2) | l=2, cửa sổ '101', +3 -> ans = 12
s = '0011100', k = 1
  r=0 thêm '0': cnt(0,1) = (1, 0) | l=0, cửa sổ '0', +1 -> ans = 1
  r=1 thêm '0': cnt(0,1) = (2, 0) | l=0, cửa sổ '00', +2 -> ans = 3
  r=2 thêm '1': cnt(0,1) = (2, 1) | l=0, cửa sổ '001', +3 -> ans = 6
  r=3 thêm '1': cnt(0,1) = (2, 2) | vi phạm -> bỏ s[0]='0', cnt = (1, 2) | l=1, cửa sổ '011', +3 -> ans = 9
  r=4 thêm '1': cnt(0,1) = (1, 3) | l=1, cửa sổ '0111', +4 -> ans = 13
  r=5 thêm '0': cnt(0,1) = (2, 3) | vi phạm -> bỏ s[1]='0', cnt = (1, 3) | l=2, cửa sổ '1110', +4 -> ans = 17
  r=6 thêm '0': cnt(0,1) = (2, 3) | vi phạm ->

In [20]:
def trace_left_driven(s: str, k: int) -> int:
    print(f"s = {s!r}, k = {k}")
    n = len(s)
    cnt = [0, 0]
    r = 0
    ans = 0
    for l in range(n):
        while r < n:
            cnt[int(s[r])] += 1
            if cnt[0] > k and cnt[1] > k:
                cnt[int(s[r])] -= 1
                print(f"  l={l}: thêm s[{r}]={s[r]!r} sẽ vi phạm -> dừng, r={r}")
                break
            r += 1
        ans += r - l
        print(f"  l={l}: đoạn hợp lệ dài nhất {s[l:r]!r} (r={r}), +{r - l} -> ans = {ans}")
        cnt[int(s[l])] -= 1
    return ans


assert trace_left_driven("10101", 1) == 12

s = '10101', k = 1
  l=0: thêm s[3]='0' sẽ vi phạm -> dừng, r=3
  l=0: đoạn hợp lệ dài nhất '101' (r=3), +3 -> ans = 3
  l=1: thêm s[4]='1' sẽ vi phạm -> dừng, r=4
  l=1: đoạn hợp lệ dài nhất '010' (r=4), +3 -> ans = 6
  l=2: đoạn hợp lệ dài nhất '101' (r=5), +3 -> ans = 9
  l=3: đoạn hợp lệ dài nhất '01' (r=5), +2 -> ans = 11
  l=4: đoạn hợp lệ dài nhất '1' (r=5), +1 -> ans = 12


Bảng trace Cách 4 với Ví dụ 1 (`s = "10101"`, `k = 1`):

| `r` | thêm | `cnt` (số 0, số 1) | vi phạm? | `l` sau co | cửa sổ | `+ (r−l+1)` | `ans` |
|---|---|---|---|---|---|---|---|
| 0 | `1` | (0, 1) | không | 0 | `1` | 1 | 1 |
| 1 | `0` | (1, 1) | không | 0 | `10` | 2 | 3 |
| 2 | `1` | (1, 2) | không (`0` chỉ `1`) | 0 | `101` | 3 | 6 |
| 3 | `0` | (2, 2) | **có** → bỏ `s[0]='1'` → (2, 1) | 1 | `010` | 3 | 9 |
| 4 | `1` | (2, 2) | **có** → bỏ `s[1]='0'` → (1, 2) | 2 | `101` | 3 | 12 |

Ba bước đầu mỗi lần cộng đủ `r + 1` chuỗi con (mọi chuỗi con kết thúc tại `r`). Hai bước sau mỗi lần *thiếu* một chuỗi con (`1010` và `0101` bị loại, cộng thêm `10101`), đúng với `3` chuỗi bị loại trong đề.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 4 (cửa sổ trượt)**: `O(n)`, ngắn, và là kỹ thuật dùng lại được cho bài II với `n` lớn.

In [21]:
class Solution:
    def countKConstraintSubstrings(self, s: str, k: int) -> int:
        cnt = [0, 0]                           # cnt[0]: số '0', cnt[1]: số '1' trong cửa sổ s[l..r]
        ans = l = 0
        for r, ch in enumerate(s):
            cnt[int(ch)] += 1                  # mở rộng bên phải
            while cnt[0] > k and cnt[1] > k:   # vi phạm: cả hai đều vượt k
                cnt[int(s[l])] -= 1            # co bên trái
                l += 1
            ans += r - l + 1                   # mọi đầu trái trong [l, r] đều hợp lệ
        return ans


sol = Solution()
for s, k, expected in examples:
    print(s, k, "->", sol.countKConstraintSubstrings(s, k), "(kỳ vọng", expected, ")")

10101 1 -> 12 (kỳ vọng 12 )
1010101 2 -> 25 (kỳ vọng 25 )
11111 1 -> 15 (kỳ vọng 15 )


## 10. Vì sao thuật toán đúng

- **Đơn điệu.** Nếu `t` hợp lệ thì mọi chuỗi con của `t` hợp lệ (đếm `0`/`1` chỉ giảm). Do đó với mỗi `r`, tập `l` hợp lệ là đoạn liên tục `[L(r), r]` (hậu tố của các đầu trái), và `L(r)` không giảm theo `r`.
- **Bất biến vòng lặp.** Sau khi xử lý `r`, `l = L(r)` là đầu trái nhỏ nhất sao cho `s[l..r]` hợp lệ, và `cnt` khớp đúng với cửa sổ `s[l..r]`. Thêm `s[r]`: nếu hợp lệ thì `l` vẫn là nhỏ nhất (vì `L(r) >= L(r-1) = l`). Nếu vi phạm, `while` đẩy `l` lên từng bước cho tới khi hợp lệ; do `L` chỉ tiến nên không bỏ sót đầu trái hợp lệ nào và không dừng quá sớm.
- **Luôn dừng.** Khi `l = r` cửa sổ chỉ có một ký tự: một trong hai số bằng `0 <= k`, nên hợp lệ (đây là chỗ cần `k >= 1`; thực ra `k >= 0` cũng đủ). Vòng `while` không bao giờ đẩy `l` vượt `r`.
- **Đếm đủ, không trùng.** Mỗi chuỗi con có duy nhất một đầu phải `r`, nên tổng `r - l + 1` qua các `r` đếm mỗi chuỗi con hợp lệ đúng một lần.
- **Đối chiếu độc lập.** Cách 1 (vét cạn theo định nghĩa) được so với các cách còn lại trên hàng nghìn chuỗi ngẫu nhiên ở mục 12.

## 11. Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn + `count` | `O(n³)` | `O(n)` |
| 2. Mở rộng phải + `break` | `O(n²)` | `O(1)` |
| 3. Tổng tiền tố + `bisect` | `O(n log n)` | `O(n)` |
| 4. Cửa sổ trượt (đầu phải điều khiển) | `O(n)` | `O(1)` |
| 5. Cửa sổ trượt (đầu trái điều khiển) | `O(n)` | `O(1)` |

Với `n <= 50` mọi cách đều nhanh, nhưng `O(n)` là bản tiêu chuẩn và mở rộng được tới `n = 10^5` (bài II).

Vì sao cửa sổ trượt `O(n)` dù có `while` lồng trong `for`? Vì `l` chỉ **tăng** từ `0` đến tối đa `n`; tổng số lần chạy thân `while` trên **toàn bộ** vòng `for` không quá `n` (phân tích khấu hao), nên tổng là `O(n) + O(n)`.

## 12. Bộ test

Gồm: ví dụ đề, biên (`n = 1`, `k = n`, chuỗi toàn `0`/toàn `1`, xen kẽ, `k = 1`), phản ví dụ cho lỗi `if` thay vì `while`, đối chiếu cả `5` cách và ngẫu nhiên có seed đúng ràng buộc (`1 <= n <= 50`, `1 <= k <= n`).

In [22]:
IMPLS = {
    "brute": count_brute,
    "extend": count_extend,
    "prefix_bisect": count_prefix_bisect,
    "window": count_window,
    "left_driven": count_left_driven,
    "Solution": Solution().countKConstraintSubstrings,
}

# 1) ví dụ đề
for s, k, expected in examples:
    for name, f in IMPLS.items():
        assert f(s, k) == expected, (name, s, k)

# 2) biên và tự dựng (đáp án tính theo công thức/biết trước)
n_all = lambda n: n * (n + 1) // 2
manual = [
    ("0", 1, 1),                                  # một ký tự
    ("1", 1, 1),
    ("0" * 50, 1, n_all(50)),                     # toàn 0: số 1 luôn là 0 <= k
    ("1" * 50, 50, n_all(50)),                    # toàn 1, k = n
    ("01" * 25, 50, n_all(50)),                   # k = n: mọi chuỗi con hợp lệ
    ("01", 1, 3),                                 # 0, 1, 01 đều hợp lệ
    ("0101", 1, 9),                               # 10 chuỗi con, chỉ "0101" (2,2) vi phạm
]
for s, k, expected in manual:
    for name, f in IMPLS.items():
        assert f(s, k) == expected, (name, s, k, f(s, k), expected)
print("ví dụ đề + biên: PASS")

ví dụ đề + biên: PASS


In [23]:
# 3) ngẫu nhiên có seed: đúng ràng buộc, mọi cách phải đồng nhất với vét cạn
rng = random.Random(3258)
for _ in range(4000):
    n = rng.randint(1, 50)
    p = rng.choice([0.5, 0.1, 0.9])            # đa dạng độ thiên lệch số 0/1
    s = "".join("1" if rng.random() < p else "0" for _ in range(n))
    k = rng.randint(1, n)
    expected = count_brute(s, k)
    for name, f in IMPLS.items():
        assert f(s, k) == expected, (name, s, k)
print("4000 test ngẫu nhiên: tất cả các cách đồng nhất — PASS")

# 4) vét cạn toàn bộ chuỗi nhị phân độ dài <= 10 với mọi k
for n in range(1, 11):
    for mask in range(1 << n):
        s = format(mask, f"0{n}b")
        for k in range(1, n + 1):
            assert count_window(s, k) == count_brute(s, k), (s, k)
print("mọi chuỗi độ dài <= 10, mọi k: PASS")

4000 test ngẫu nhiên: tất cả các cách đồng nhất — PASS
mọi chuỗi độ dài <= 10, mọi k: PASS


In [24]:
# 5) phản ví dụ cho lỗi "if thay while" (xem mục 13): hàm sai phải KHÁC vét cạn ở ít nhất một chuỗi
def count_if_bug(s: str, k: int) -> int:
    cnt = [0, 0]
    ans = l = 0
    for r, ch in enumerate(s):
        cnt[int(ch)] += 1
        if cnt[0] > k and cnt[1] > k:          # SAI: chỉ co một lần
            cnt[int(s[l])] -= 1
            l += 1
        ans += r - l + 1
    return ans


diffs = [(s, k) for n in range(1, 9) for mask in range(1 << n)
         for s in [format(mask, f"0{n}b")] for k in range(1, n + 1)
         if count_if_bug(s, k) != count_brute(s, k)]
print("số cặp (s, k) mà bản 'if' sai:", len(diffs), "| ví dụ ngắn nhất:", min(diffs, key=lambda x: len(x[0])))
assert diffs

số cặp (s, k) mà bản 'if' sai: 406 | ví dụ ngắn nhất: ('00011', 1)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Nhầm `and` / `or` ở điều kiện co cửa sổ

Hợp lệ là `zeros <= k` **hoặc** `ones <= k`. Vì vậy **vi phạm** là `zeros > k` **và** `ones > k`. Nếu viết `or` ở `while`, ta co cửa sổ ngay khi *một* trong hai vượt `k` — co quá tay, đếm thiếu.

In [25]:
def sai_or(s, k):
    cnt = [0, 0]
    ans = l = 0
    for r, ch in enumerate(s):
        cnt[int(ch)] += 1
        while cnt[0] > k or cnt[1] > k:        # SAI: phải là "and"
            cnt[int(s[l])] -= 1
            l += 1
        ans += r - l + 1
    return ans

for s, k, expected in examples:
    print(f"s={s!r}, k={k}: sai={sai_or(s, k)}, đúng={expected}")

s='10101', k=1: sai=9, đúng=12
s='1010101', k=2: sai=22, đúng=25
s='11111', k=1: sai=5, đúng=15


Sửa: `while cnt[0] > k and cnt[1] > k:` — chỉ co khi **cả hai** cùng vượt `k`.

### Lỗi 2: Cộng `r - l` thay vì `r - l + 1`

Với `l = r` có **một** chuỗi con `s[r..r]`. Công thức `r - l = 0` bỏ qua nó. Số chỉ số trong đoạn `[l, r]` là `r - l + 1` (quy tắc "hàng rào": đếm cả hai đầu).

In [26]:
def sai_off_by_one(s, k):
    cnt = [0, 0]
    ans = l = 0
    for r, ch in enumerate(s):
        cnt[int(ch)] += 1
        while cnt[0] > k and cnt[1] > k:
            cnt[int(s[l])] -= 1
            l += 1
        ans += r - l                           # SAI: thiếu + 1
    return ans

for s, k, expected in examples:
    print(f"s={s!r}, k={k}: sai={sai_off_by_one(s, k)}, đúng={expected}")

s='10101', k=1: sai=7, đúng=12
s='1010101', k=2: sai=18, đúng=25
s='11111', k=1: sai=10, đúng=15


### Lỗi 3: Co cửa sổ bằng `if` thay vì `while`

Bỏ một ký tự bên trái **chưa chắc** làm hết vi phạm: nếu ký tự bỏ đi là loại **không** gây vi phạm thì `cnt` vẫn vượt. Cần co **lặp lại** cho tới khi hợp lệ. Mục 12 (test số 5) đã đếm được số trường hợp sai; dưới đây là một ví dụ cụ thể ngắn nhất.

In [27]:
s, k = "00011", 1
print("đúng (vét cạn):", count_brute(s, k))
print("dùng if:", count_if_bug(s, k))
print("dùng while:", count_window(s, k))

đúng (vét cạn): 13
dùng if: 14
dùng while: 13


### Lỗi 4: Quên trừ đếm khi `l` tiến (hoặc trừ sai ký tự)

Khi `l += 1` mà **không** `cnt[int(s[l])] -= 1` thì `cnt` đếm nhiều hơn thực tế và vòng `while` có thể chạy tới khi `l` vượt `r` (lặp vô hạn / lỗi chỉ số). Trừ **trước** khi tăng `l` (vì `s[l]` là ký tự rời cửa sổ).

In [28]:
def dung_tru_dem(s, k):
    cnt = [0, 0]
    ans = l = 0
    for r, ch in enumerate(s):
        cnt[int(ch)] += 1
        while cnt[0] > k and cnt[1] > k:
            cnt[int(s[l])] -= 1          # trừ ký tự rời cửa sổ TRƯỚC khi tăng l
            l += 1
        ans += r - l + 1
    return ans

# Minh họa lỗi tăng l trước rồi trừ nhầm ký tự s[l] (đã là ký tự kế tiếp):
def sai_tru_nham(s, k):
    cnt = [0, 0]
    ans = l = 0
    for r, ch in enumerate(s):
        cnt[int(ch)] += 1
        steps = 0
        while cnt[0] > k and cnt[1] > k and steps < 100:
            l += 1
            cnt[int(s[l])] -= 1          # SAI: trừ ký tự kế tiếp, không phải ký tự vừa rời đi
            steps += 1
        ans += r - l + 1
    return ans

print("sai:", sai_tru_nham("10101", 1), "| đúng:", dung_tru_dem("10101", 1))

sai: 13 | đúng: 12


### Lỗi 5: Đếm chỉ theo một loại ký tự

Chỉ kiểm tra số `0` (hoặc chỉ số `1`) bỏ qua vế còn lại của "hoặc". Ví dụ 3 (`"11111"`) sẽ cho kết quả sai nếu chỉ xét số `1 <= k`: chuỗi dài hơn `k` toàn `1` bị loại oan dù số `0 = 0 <= k`.

In [29]:
def sai_mot_loai(s, k):
    cnt = [0, 0]
    ans = l = 0
    for r, ch in enumerate(s):
        cnt[int(ch)] += 1
        while cnt[1] > k:                      # SAI: chỉ xét số 1, quên vế "số 0 <= k"
            cnt[int(s[l])] -= 1
            l += 1
        ans += r - l + 1
    return ans

for s, k, expected in examples:
    print(f"s={s!r}, k={k}: sai={sai_mot_loai(s, k)}, đúng={expected}")

s='10101', k=1: sai=10, đúng=12
s='1010101', k=2: sai=23, đúng=25
s='11111', k=1: sai=5, đúng=15


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm số chuỗi con / đoạn con thỏa điều kiện" + điều kiện đơn điệu (con của hợp lệ cũng hợp lệ)** ⇒ nghĩ ngay tới **cửa sổ trượt**, đáp án cộng `r - l + 1` mỗi bước.
- **Điều kiện "hoặc" khó** thì viết **phủ định** (De Morgan): *hợp lệ* = A hoặc B ⇒ *vi phạm* = (không A) **và** (không B). Vòng `while` của cửa sổ luôn xoay quanh điều kiện **vi phạm**.
- **`r - l + 1`:** mỗi đầu phải `r` đóng góp số chuỗi con hợp lệ **kết thúc** ở `r`. Cùng mẹo xuất hiện ở bài "đếm số mảng con tích nhỏ hơn k", "đếm chuỗi con không lặp ký tự"...
- **Kiểm tra bằng vét cạn nhỏ:** với `n <= 50` luôn viết thêm bản `O(n³)` theo đúng định nghĩa để đối chiếu — dễ nhất để bắt lỗi `and`/`or` và lệch một đơn vị.
- **`while` chứ không `if`** khi co cửa sổ — bỏ một phần tử chưa chắc làm hết vi phạm.
- **Đếm hai loại bằng mảng `[0, 0]`** với `int(ch)` làm chỉ số: gọn hơn `dict` khi khóa là `0`/`1`.

## 15. Biến thể của bài

1. **Bài II (Hard-ish).** Cùng đề nhưng cho thêm **truy vấn** `[l, r]` và hỏi số chuỗi con hợp lệ **nằm trong** `s[l..r]`; `n` tới `10^5`. Cần tính trước `L(r)` (từ cửa sổ trượt) rồi dùng tổng tiền tố và tìm kiếm nhị phân.
2. **Điều kiện "và".** Chuỗi hợp lệ nếu `zeros <= k` **và** `ones <= k`: tính đơn điệu vẫn đúng; vi phạm khi **một** trong hai vượt `k` (co khi `or`).
3. **Chuỗi con dài nhất hợp lệ.** Giữ cửa sổ và theo dõi `max(r - l + 1)` thay vì cộng.
4. **Ngưỡng khác nhau.** Hợp lệ nếu `zeros <= a` hoặc `ones <= b`: chỉ đổi điều kiện co thành `zeros > a and ones > b`.
5. **Bảng chữ cái lớn hơn.** Với `m` loại ký tự, cần đếm bằng `Counter`/`dict` và định nghĩa "vi phạm" theo bài (ví dụ "tối đa `k` ký tự khác nhau").

In [30]:
# Minh họa biến thể 2 (điều kiện "và") và biến thể 3 (độ dài lớn nhất hợp lệ)
def count_and_variant(s: str, k: int) -> int:
    cnt = [0, 0]
    ans = l = 0
    for r, ch in enumerate(s):
        cnt[int(ch)] += 1
        while cnt[0] > k or cnt[1] > k:        # vi phạm khi MỘT trong hai vượt k
            cnt[int(s[l])] -= 1
            l += 1
        ans += r - l + 1
    return ans


def longest_valid(s: str, k: int) -> int:
    cnt = [0, 0]
    best = l = 0
    for r, ch in enumerate(s):
        cnt[int(ch)] += 1
        while cnt[0] > k and cnt[1] > k:
            cnt[int(s[l])] -= 1
            l += 1
        best = max(best, r - l + 1)
    return best


print(count_and_variant("10101", 1))     # chỉ tính chuỗi có cả số 0 và số 1 đều <= 1
print(longest_valid("10101", 1), longest_valid("1010101", 2), longest_valid("11111", 1))   # 3 5 5

9
3 5 5


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn + `count` | Duyệt mọi `(l, r)`, đếm lại | `O(n³)` | `O(n)` | Bản chuẩn để đối chiếu; `n` rất nhỏ |
| 2. Mở rộng phải + `break` | Cập nhật đếm `O(1)`, dừng khi vi phạm | `O(n²)` | `O(1)` | Dễ viết, `n <= 1000` |
| 3. Tiền tố + `bisect` | Đầu phải xa nhất bằng tìm nhị phân | `O(n log n)` | `O(n)` | Khi cần trả lời **nhiều truy vấn** trên đoạn |
| 4. Cửa sổ trượt (phải điều khiển) | `ans += r - l + 1`, co `l` bằng `while` | `O(n)` | `O(1)` | **Cách nộp chuẩn** |
| 5. Cửa sổ trượt (trái điều khiển) | Với mỗi `l`, `r` chỉ đi tới | `O(n)` | `O(1)` | Khi nghĩ theo "đoạn dài nhất bắt đầu từ `l`" |

Lời khuyên: nộp **Cách 4**; giữ **Cách 1** làm chuẩn kiểm thử khi bạn thay đổi điều kiện bài.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"1010".count("0")`, `"0110"[1:3]`, `list(accumulate([1, 0, 1], initial=0))`, `Solution().countKConstraintSubstrings("01", 1)`.

**Bài 2 (tính tay).** Với `s = "0110"`, `k = 1`: liệt kê mọi `10` chuỗi con, đánh dấu cái nào thỏa k-constraint, rồi cho biết đáp án. Sau đó chạy tay cửa sổ trượt (`cnt`, `l`, `ans` sau mỗi `r`).

**Bài 3 (điền TODO).** Hoàn thành `count_todo(s, k)` theo Cách 4 (cửa sổ trượt).

In [31]:
def count_todo(s, k):
    # TODO 1: cnt = [0, 0]; ans = l = 0
    # TODO 2: for r, ch in enumerate(s): cnt[int(ch)] += 1
    # TODO 3: while cnt[0] > k and cnt[1] > k: trừ cnt[int(s[l])], tăng l
    # TODO 4: ans += r - l + 1
    # TODO 5: trả về ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Không nhìn lời giải, viết `count_invalid(s, k)` đếm số chuỗi con **vi phạm** k-constraint (ngược lại với đề) bằng cách lấy tổng `n(n+1)/2` trừ đáp án; sau đó viết bản **độc lập** dùng vét cạn rồi so sánh hai bản.

**Bài 5 (biến thể / mini challenge).** (a) Viết `longest_k_substring(s, k)` trả về độ dài chuỗi con hợp lệ **dài nhất**. (b) Viết `count_at_most(s, a, b)` đếm số chuỗi con có `zeros <= a` **hoặc** `ones <= b`. (c) Viết `count_exactly_one_side(s, k)` đếm chuỗi con mà **đúng một** trong hai điều kiện thỏa (`zeros <= k` XOR `ones <= k`); gợi ý: dùng vét cạn rồi tối ưu bằng cửa sổ trượt cho hai điều kiện riêng.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"1010".count("0") = 2`; `"0110"[1:3] = "11"`; `accumulate(..., initial=0)` bắt đầu bằng `0` rồi cộng dồn: `[0, 1, 1, 2]`. Chuỗi `"01"` có `3` chuỗi con (`0`, `1`, `01`) và `k = 1` nên đều thỏa. Bài 2: chuỗi con có `(0, 1)`, `(1, 0)`... chỉ cần đếm `0` và `1` rồi so `<= 1`.
- **Mức 2 (vừa):** Bài 3: dùng đúng khung ở mục 9; nhớ `while` chứ không `if`. Bài 4: tổng số chuỗi con là `n * (n + 1) // 2`.
- **Mức 3 (sâu):** Bài 5(a): giữ `best = max(best, r - l + 1)`. 5(b): vi phạm khi `zeros > a and ones > b`. 5(c): với mỗi cặp, `zeros <= k` XOR `ones <= k` nghĩa là `(zeros <= k) != (ones <= k)`; có thể đếm bằng `A + B - 2 * AB` với `A` = số chuỗi con `zeros <= k`, `B` = số chuỗi con `ones <= k`, `AB` = số chuỗi con cả hai (đáp án bài gốc là `A + B - AB`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert "1010".count("0") == 2
assert "0110"[1:3] == "11"
assert list(accumulate([1, 0, 1], initial=0)) == [0, 1, 1, 2]
assert Solution().countKConstraintSubstrings("01", 1) == 3

# Bài 2: s = "0110", k = 1 -> chuỗi con: 0,01,011,0110,1,11,110,1,10,0
# (số0,số1): 0:(1,0) ok | 01:(1,1) ok | 011:(1,2) ok | 0110:(2,2) KHÔNG | 1 ok | 11:(0,2) ok | 110:(1,2) ok | 1 ok | 10 ok | 0 ok
assert count_brute("0110", 1) == 9
assert Solution().countKConstraintSubstrings("0110", 1) == 9


# Bài 3
def count_todo_ans(s, k):
    cnt = [0, 0]
    ans = l = 0
    for r, ch in enumerate(s):
        cnt[int(ch)] += 1
        while cnt[0] > k and cnt[1] > k:
            cnt[int(s[l])] -= 1
            l += 1
        ans += r - l + 1
    return ans


for s, k, expected in examples:
    assert count_todo_ans(s, k) == expected


# Bài 4
def count_invalid(s, k):
    n = len(s)
    return n * (n + 1) // 2 - Solution().countKConstraintSubstrings(s, k)


def count_invalid_brute(s, k):
    n = len(s)
    return sum(1 for l in range(n) for r in range(l, n)
               if s[l:r + 1].count("0") > k and s[l:r + 1].count("1") > k)


rng = random.Random(42)
for _ in range(500):
    n = rng.randint(1, 30)
    s = "".join(rng.choice("01") for _ in range(n))
    k = rng.randint(1, n)
    assert count_invalid(s, k) == count_invalid_brute(s, k)
assert count_invalid("10101", 1) == 3


# Bài 5
def longest_k_substring(s, k):
    cnt = [0, 0]
    best = l = 0
    for r, ch in enumerate(s):
        cnt[int(ch)] += 1
        while cnt[0] > k and cnt[1] > k:
            cnt[int(s[l])] -= 1
            l += 1
        best = max(best, r - l + 1)
    return best


def count_at_most(s, a, b):
    cnt = [0, 0]
    ans = l = 0
    for r, ch in enumerate(s):
        cnt[int(ch)] += 1
        while cnt[0] > a and cnt[1] > b:       # vi phạm: zeros > a VÀ ones > b
            cnt[int(s[l])] -= 1
            l += 1
        ans += r - l + 1
    return ans


def count_exactly_one_side(s, k):
    def at_most(c):                            # số chuỗi con có số ký tự c <= k (cửa sổ trượt một điều kiện)
        cnt = 0
        ans = l = 0
        for r, ch in enumerate(s):
            cnt += (ch == c)
            while cnt > k:
                cnt -= (s[l] == c)
                l += 1
            ans += r - l + 1
        return ans

    A, B = at_most("0"), at_most("1")
    AB = count_and_variant(s, k)                # cả hai điều kiện cùng thỏa (biến thể "và")
    return A + B - 2 * AB


def exactly_one_brute(s, k):
    n = len(s)
    return sum(1 for l in range(n) for r in range(l, n)
               if (s[l:r + 1].count("0") <= k) != (s[l:r + 1].count("1") <= k))


assert longest_k_substring("10101", 1) == 3 and longest_k_substring("1010101", 2) == 5 and longest_k_substring("11111", 1) == 5
assert count_at_most("10101", 1, 1) == 12
rng = random.Random(99)
for _ in range(500):
    n = rng.randint(1, 30)
    s = "".join(rng.choice("01") for _ in range(n))
    k = rng.randint(1, n)
    a, b = rng.randint(0, n), rng.randint(0, n)
    brute = sum(1 for l in range(n) for r in range(l, n)
                if s[l:r + 1].count("0") <= a or s[l:r + 1].count("1") <= b)
    assert count_at_most(s, a, b) == brute
    assert count_exactly_one_side(s, k) == exactly_one_brute(s, k)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cửa sổ trượt đếm chuỗi con hợp lệ kết thúc tại mỗi `r`)

```python
class Solution:
    def countKConstraintSubstrings(self, s: str, k: int) -> int:
        cnt = [0, 0]
        ans = l = 0
        for r, ch in enumerate(s):
            cnt[int(ch)] += 1
            while cnt[0] > k and cnt[1] > k:
                cnt[int(s[l])] -= 1
                l += 1
            ans += r - l + 1
        return ans
```

**Checklist**

- [ ] *Vi phạm* viết bằng **`and`** (`cnt[0] > k and cnt[1] > k`), vì hợp lệ là "hoặc"?
- [ ] Co cửa sổ bằng **`while`**, không phải `if`?
- [ ] Mỗi lần `l += 1` đều **trừ đúng** `cnt[int(s[l])]` trước đó?
- [ ] Cộng **`r - l + 1`** (không phải `r - l`) sau khi cửa sổ đã hợp lệ?
- [ ] Đã thử `3` ví dụ đề (`12`, `25`, `15`) và biên (`n = 1`, `k = n`)?
- [ ] Đã đối chiếu với bản vét cạn trên chuỗi ngẫu nhiên nhỏ?
- [ ] Skeleton đúng `class Solution` / `def countKConstraintSubstrings(self, s: str, k: int) -> int`?